In [ ]:
import pubchempy as pcp, pandas as pd, requests, time

In [ ]:
# Set input/output file names here
INPUT_FILE = 'PubChem_input1.xlsx'
OUTPUT_FILE = 'PubChem_output1.xlsx'

In [ ]:
# Read in dataframe from excel
df = pd.read_excel(INPUT_FILE)
df.head(50)

In [ ]:
# Put columns in list
cas_list = df['CAS#'].tolist()
name_list = df['Name'].tolist()

In [ ]:
# Search names using pubchempy and get a list of CIDs
cid_lst = []
total = len(name_list)

for i, x in enumerate(name_list, 1):
    print(f'  Looking up name [{i}/{total}]: {x}...')
    try:
        cids = pcp.get_cids(x, 'name')
        cid = cids[0] if cids else 0
    except Exception as e:
        print(f'    Warning: name lookup failed ({e})')
        cid = 0
    cid_lst.append(cid)

print('Done.')

In [ ]:
# Search CAS numbers as names using pubchempy and get list of CIDs
cas_cids = []
total = len(cas_list)

for i, x in enumerate(cas_list, 1):
    x_str = str(x)
    print(f'  Looking up CAS [{i}/{total}]: {x_str}...')
    if x_str == 'nan':
        cas_cids.append(0)
        continue
    try:
        cids = pcp.get_cids(x_str, 'name')
        if cids:
            cid = cids[0]
        else:
            cids = pcp.get_cids('CAS-' + x_str, 'name')
            cid = cids[0] if cids else 0
    except Exception as e:
        print(f'    Warning: CAS lookup failed ({e})')
        cid = 0
    cas_cids.append(cid)

print('Done.')

In [ ]:
df = pd.DataFrame()
df['Labeled Names'] = name_list
df['Labeled CAS'] = cas_list
df['CID from Name'] = cid_lst
df['CID from CAS #'] = cas_cids

In [ ]:
df.head(50)

In [ ]:
# Function to extract data from PubChem JSON response
def get_pc_data(cid, retries=3, backoff=5):
    """Fetch compound metadata from PubChem by CID.

    Parameters
    ----------
    cid : int
        PubChem Compound ID to look up.
    retries : int, optional
        Number of retry attempts on server errors (503/429), by default 3.
    backoff : int, optional
        Base wait time in seconds between retries (multiplied by attempt
        number), by default 5.

    Returns
    -------
    pc_name : str
        Official PubChem compound name.
    pc_cas : str
        Primary CAS number registered in PubChem.
    other_cas : str
        Alternate CAS numbers joined by newlines, or 'None'.
    rel_cas : str
        Related CAS numbers joined by newlines, or 'None'.
    new_syn_lst : str
        Up to 5 deduplicated depositor-supplied synonyms joined by newlines.
    ghs_pict : str
        GHS pictogram labels joined by newlines, or 'None'.
    ghs_haz : str
        Sorted GHS H-statement codes and text joined by newlines, or 'None'.

    Raises
    ------
    requests.HTTPError
        If the PubChem API returns a non-200 status after all retries.
    """
    pc_cas = ''
    other_cas = []
    rel_cas = []
    syn_lst = []
    ghs_pict = []
    ghs_haz = []

    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON'
    for attempt in range(1, retries + 1):
        resp = requests.get(url)
        if resp.status_code == 200:
            break
        if resp.status_code in (503, 429) and attempt < retries:
            wait = backoff * attempt
            print(f'    Server busy (HTTP {resp.status_code}), retrying in {wait}s...')
            time.sleep(wait)
        else:
            resp.raise_for_status()

    data = resp.json()
    pc_name = data['Record']['RecordTitle']

    for a in data['Record'].get('Section', []):
        if a['TOCHeading'] == 'Names and Identifiers':
            for b in a.get('Section', []):
                if b['TOCHeading'] == 'Other Identifiers':
                    for c in b.get('Section', []):
                        if c['TOCHeading'] == 'CAS':
                            try:
                                pc_cas = c['Information'][0]['Value']['StringWithMarkup'][0]['String']
                            except (KeyError, IndexError):
                                pass
                            for d in c.get('Information', []):
                                try:
                                    val = d['Value']['StringWithMarkup'][0]['String']
                                except (KeyError, IndexError):
                                    continue
                                name = d.get('Name', '')
                                if name == 'Other CAS':
                                    other_cas.append(val)
                                elif name == 'Related CAS':
                                    rel_cas.append(val)
                                elif not name and val != pc_cas:
                                    other_cas.append(val)

                elif b['TOCHeading'] == 'Synonyms':
                    for c in b.get('Section', []):
                        if c['TOCHeading'] == 'Depositor-Supplied Synonyms':
                            try:
                                entries = c['Information'][0]['Value']['StringWithMarkup']
                                for e in entries:
                                    syn_lst.append(e['String'])
                            except (KeyError, IndexError):
                                pass

        elif a['TOCHeading'] == 'Safety and Hazards':
            for b in a.get('Section', []):
                if b['TOCHeading'] == 'Hazards Identification':
                    for c in b.get('Section', []):
                        if c['TOCHeading'] == 'GHS Classification':
                            for d in c.get('Information', []):
                                name = d.get('Name', '')
                                try:
                                    if name == 'Pictogram(s)':
                                        for e in d['Value']['StringWithMarkup'][0]['Markup']:
                                            ghs_pict.append(e['Extra'])
                                    elif name == 'GHS Hazard Statements':
                                        for e in d['Value']['StringWithMarkup']:
                                            ghs_haz.append(e['String'])
                                except (KeyError, IndexError):
                                    pass

    # --- Format other_cas ---
    other_cas = sorted(set(other_cas))
    other_cas = 'None' if not other_cas else '\n'.join(other_cas)

    # --- Format rel_cas ---
    rel_cas = sorted(set(rel_cas))
    rel_cas = 'None' if not rel_cas else '\n'.join(rel_cas)

    # --- Deduplicate synonyms (case-insensitive), keep top 5 ---
    seen = set()
    deduped_syns = []
    for s in syn_lst:
        if s.lower() not in seen:
            seen.add(s.lower())
            deduped_syns.append(s)
    new_syn_lst = 'None' if not deduped_syns else '\n'.join(deduped_syns[:5])

    # --- Format GHS pictograms ---
    ghs_pict = sorted(set(ghs_pict))
    ghs_pict = 'None' if not ghs_pict else '\n'.join(ghs_pict)

    # --- Format GHS hazard statements (H-codes only, sorted) ---
    ghs_haz = [x for x in ghs_haz if x and x[0] == 'H']
    ghs_haz = sorted(set(ghs_haz))
    ghs_haz = 'None' if not ghs_haz else '\n'.join(ghs_haz)

    return pc_name, pc_cas, other_cas, rel_cas, new_syn_lst, ghs_pict, ghs_haz

In [ ]:
# Extract data and add to lists
# Falls back to CID from Name when CAS lookup returned nothing
pubchem_names = []
pubchem_cas = []
other_cas_numbers = []
rel_cas_numbers = []
synonyms = []
ghs_pictograms = []
ghs_hazards = []

rows = list(zip(df['CID from CAS #'].tolist(), df['CID from Name'].tolist(), df['Labeled Names'].tolist()))
total = len(rows)

for i, (cid_cas, cid_name, label) in enumerate(rows, 1):
    cid = cid_cas if cid_cas != 0 else cid_name
    print(f'  Fetching metadata [{i}/{total}]: {label} (CID={cid})...')
    if cid != 0:
        try:
            pc_name, pc_cas, other_cas, rel_cas, new_syn_lst, ghs_pict, ghs_haz = get_pc_data(cid)
            pubchem_names.append(pc_name)
            pubchem_cas.append(pc_cas)
            other_cas_numbers.append(other_cas)
            rel_cas_numbers.append(rel_cas)
            synonyms.append(new_syn_lst)
            ghs_pictograms.append(ghs_pict)
            ghs_hazards.append(ghs_haz)
        except Exception as e:
            print(f'    Error fetching CID {cid}: {e}')
            pubchem_names.append('Error')
            pubchem_cas.append('')
            other_cas_numbers.append('')
            rel_cas_numbers.append('')
            synonyms.append('')
            ghs_pictograms.append('')
            ghs_hazards.append('')
    else:
        pubchem_names.append('Not Found')
        pubchem_cas.append('')
        other_cas_numbers.append('')
        rel_cas_numbers.append('')
        synonyms.append('')
        ghs_pictograms.append('')
        ghs_hazards.append('')

print('Done.')

In [ ]:
# Add results to dataframe
df['PubChem Name'] = pubchem_names
df['PubChem CAS #'] = pubchem_cas
df['Other CAS #s'] = other_cas_numbers
df['Related CAS #s'] = rel_cas_numbers
df['Top 5 Chem Names'] = synonyms
df['GHS Pictogram Names'] = ghs_pictograms
df['GHS Hazard Statements'] = ghs_hazards

In [ ]:
df.head(50)

In [ ]:
# Write results to Excel
with pd.ExcelWriter(OUTPUT_FILE) as writer:
    df.to_excel(writer, index=False)
print(f'Saved to {OUTPUT_FILE}')